# 260415-2: Char-level Name Classification with RNN / LSTM

260415-1 에서 RNN / LSTM / GRU forward 를 직접 구현했습니다.
이번 실습에서는 PyTorch 의 `nn.RNN` / `nn.LSTM` 모듈을 그대로 사용해 실제 task —
**이름 (성씨) 의 언어 분류** — 에 적용하고, 두 모델을 여러 각도에서 비교합니다.

## 학습 목표
- 18 개 언어로 라벨링된 이름 dataset 을 자동 다운로드 + 전처리 + train/test 분할
- `nn.RNN` 기반 char-level 분류기 (`CharRNN`) 정의 및 학습
- 학습된 모델을 test set 으로 평가하는 함수 작성
- 같은 구조를 `nn.LSTM` 으로 바꾼 분류기 (`CharLSTM`) 와 다음 관점에서 비교
  - 학습 곡선 (loss / running accuracy)
  - test set confusion matrix
  - per-category accuracy
  - 이름 길이별 accuracy
  - 두 모델이 서로 다르게 예측한 이름 목록

## 구성
1. 데이터셋 다운로드 + 전처리 + train/test 분할
2. CharRNN 분류기 정의
3. 학습 루프
4. Test 평가 (confusion matrix, per-category)
5. CharLSTM 과 비교
6. 길이별 / disagreement 분석


### Recap

260415-1 에서 RNN / LSTM / GRU 의 forward 를 numpy 로 직접 구현하고 PyTorch 와 수치 비교했습니다.
또 같은 시퀀스 길이에서 RNN 의 $\|h_0.\text{grad}\|$ 가 LSTM/GRU 보다 훨씬 빠르게 사라지는 것을 보았습니다.
이번 실습은 그 차이가 **실제 task** 에서 어떤 식으로 나타나는지를 보는 것이 목표입니다.
구현은 `nn.RNN` / `nn.LSTM` 모듈을 그대로 사용합니다.


In [ ]:
# Colab environment setup
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    import subprocess
    gpu_info = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
    if gpu_info.returncode == 0:
        print('GPU available:')
        print(gpu_info.stdout.split('\n')[8])
    else:
        print('No GPU detected. Go to Runtime > Change runtime type > GPU')


In [ ]:
import os
import io
import time
import math
import random
import string
import unicodedata
import urllib.request
import zipfile
from collections import defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams['figure.figsize'] = (10.0, 6.0)

if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')
print(f'Using device: {device}')


In [ ]:
# LSTM cell diagram (visual recap from 260415-1, provided — just run)
from matplotlib.patches import FancyBboxPatch, Circle

GREEN  = '#2E7D32'
BLUE   = '#1565C0'
ORANGE = '#EF6C00'
GREY   = '#546E7A'
BG     = '#FAFBFC'

fig, ax = plt.subplots(figsize=(14, 7.2))
ax.set_xlim(0, 14); ax.set_ylim(0, 7.2); ax.axis('off')

ax.add_patch(FancyBboxPatch((0.2, 0.25), 13.6, 6.75, boxstyle='round,pad=0.15',
                             facecolor=BG, edgecolor='#CFD8DC', lw=1.2))
ax.text(7.0, 6.7, 'LSTM cell — one timestep', ha='center', fontsize=15,
        fontweight='bold', color='#263238')

# ── Cell state highway (top) ─────────────────────────────────────────────
C_Y = 5.55
ax.plot([0.55, 12.9], [C_Y, C_Y], color=GREEN, lw=5, solid_capstyle='round', zorder=2)
ax.text(0.35, C_Y, r'$c_{t-1}$', fontsize=14, ha='right', va='center',
        color=GREEN, fontweight='bold')
ax.text(13.1, C_Y, r'$c_t$', fontsize=14, ha='left', va='center',
        color=GREEN, fontweight='bold')
ax.text(6.7, 6.04, 'cell state  (long-term memory, no nonlinearity on the highway)',
        ha='center', fontsize=10, color=GREEN, style='italic')


def op_circle(cx, cy, symbol, color, r=0.32):
    ax.add_patch(Circle((cx, cy), r, facecolor='white', edgecolor=color, lw=2.5, zorder=4))
    ax.text(cx, cy, symbol, ha='center', va='center',
            fontsize=18, color=color, zorder=5)


FORGET_X = 3.0
ADD_X    = 6.3
BRANCH_X = 9.3
OUT_X    = 10.9

op_circle(FORGET_X, C_Y, r'$\times$', GREEN)
op_circle(ADD_X, C_Y, r'$+$', GREEN)
ax.plot([BRANCH_X], [C_Y], marker='o', color=GREEN, markersize=8, zorder=4)

# ── h_t branch (off-highway) ─────────────────────────────────────────────
TANH_Y = 4.35
ax.annotate('', xy=(BRANCH_X, TANH_Y + 0.35), xytext=(BRANCH_X, C_Y - 0.02),
            arrowprops=dict(arrowstyle='->', color=GREEN, lw=2.2))

ax.add_patch(FancyBboxPatch((BRANCH_X - 0.45, TANH_Y - 0.35), 0.9, 0.7,
                             boxstyle='round,pad=0.05',
                             facecolor='#FFF3E0', edgecolor=ORANGE, lw=2, zorder=3))
ax.text(BRANCH_X, TANH_Y, r'$\tanh$', ha='center', va='center', fontsize=11, color=ORANGE)

ax.annotate('', xy=(OUT_X - 0.33, TANH_Y), xytext=(BRANCH_X + 0.45, TANH_Y),
            arrowprops=dict(arrowstyle='->', color=GREEN, lw=2))
op_circle(OUT_X, TANH_Y, r'$\times$', BLUE)

ax.annotate('', xy=(13.0, TANH_Y), xytext=(OUT_X + 0.33, TANH_Y),
            arrowprops=dict(arrowstyle='->', color=BLUE, lw=2.2))
ax.text(13.15, TANH_Y, r'$h_t$', ha='left', va='center',
        fontsize=14, color=BLUE, fontweight='bold')

# ── Gate activation boxes (middle row) ───────────────────────────────────
GATE_Y = 2.7
GATE_W = 1.25
GATE_H = 0.85


def gate_box(cx, inner_label, outer_label):
    ax.add_patch(FancyBboxPatch((cx - GATE_W / 2, GATE_Y - GATE_H / 2),
                                 GATE_W, GATE_H, boxstyle='round,pad=0.06',
                                 facecolor='#E3F2FD', edgecolor=BLUE, lw=2))
    ax.text(cx, GATE_Y + 0.05, inner_label, ha='center', va='center',
            fontsize=13, fontweight='bold', color=BLUE)
    ax.text(cx, GATE_Y - 0.72, outer_label, ha='center', va='center',
            fontsize=10.5, color=BLUE, style='italic')


gate_box(FORGET_X, r'$\sigma$', r'$f_t$  forget')
gate_box(5.0, r'$\sigma$', r'$i_t$  input')
gate_box(7.0, r'$\tanh$', r'$g_t$  candidate')
gate_box(OUT_X, r'$\sigma$', r'$o_t$  output')

# ── Gate → highway/branch arrows ─────────────────────────────────────────
ax.annotate('', xy=(FORGET_X, C_Y - 0.33), xytext=(FORGET_X, GATE_Y + GATE_H / 2),
            arrowprops=dict(arrowstyle='->', color=BLUE, lw=1.8))

IX_G_Y = 4.5
ax.add_patch(Circle((ADD_X, IX_G_Y), 0.24, facecolor='white', edgecolor=BLUE, lw=2, zorder=4))
ax.text(ADD_X, IX_G_Y, r'$\times$', ha='center', va='center',
        fontsize=15, color=BLUE, zorder=5)

ax.annotate('', xy=(ADD_X - 0.15, IX_G_Y - 0.12),
            xytext=(5.0 + 0.28, GATE_Y + GATE_H / 2),
            arrowprops=dict(arrowstyle='->', color=BLUE, lw=1.6))
ax.annotate('', xy=(ADD_X + 0.15, IX_G_Y - 0.12),
            xytext=(7.0 - 0.28, GATE_Y + GATE_H / 2),
            arrowprops=dict(arrowstyle='->', color=BLUE, lw=1.6))
ax.annotate('', xy=(ADD_X, C_Y - 0.33),
            xytext=(ADD_X, IX_G_Y + 0.25),
            arrowprops=dict(arrowstyle='->', color=GREEN, lw=2))

ax.annotate('', xy=(OUT_X, TANH_Y - 0.33),
            xytext=(OUT_X, GATE_Y + GATE_H / 2),
            arrowprops=dict(arrowstyle='->', color=BLUE, lw=1.8))

# ── Input merge strip (bottom) ───────────────────────────────────────────
MERGE_Y = 1.15
ax.add_patch(FancyBboxPatch((1.8, MERGE_Y - 0.32), 9.3, 0.64,
                             boxstyle='round,pad=0.06',
                             facecolor='#ECEFF1', edgecolor=GREY, lw=1.5))
ax.text(6.45, MERGE_Y,
        r'$W\cdot[\,x_t,\;h_{t-1}\,]\;+\;b$   '
        r'$\longrightarrow$  $(a_f, a_i, a_g, a_o)$',
        ha='center', va='center', fontsize=11, color='#263238')

ax.text(0.65, 0.5, r'$x_t$', ha='center', va='center',
        fontsize=14, color=ORANGE, fontweight='bold')
ax.annotate('', xy=(1.75, MERGE_Y), xytext=(0.65, 0.75),
            arrowprops=dict(arrowstyle='->', color=ORANGE, lw=2))

ax.text(0.35, MERGE_Y, r'$h_{t-1}$', ha='right', va='center',
        fontsize=15, color=BLUE)
ax.annotate('', xy=(1.75, MERGE_Y), xytext=(0.75, MERGE_Y),
            arrowprops=dict(arrowstyle='->', color=BLUE, lw=2))

for gx in [FORGET_X, 5.0, 7.0, OUT_X]:
    ax.annotate('', xy=(gx, GATE_Y - GATE_H / 2),
                xytext=(gx, MERGE_Y + 0.32),
                arrowprops=dict(arrowstyle='->', color=GREY, lw=1.2,
                                linestyle='--', alpha=0.75))

plt.tight_layout(); plt.show()


## 1. 데이터셋 준비

**Task**: 사람 이름 (예: "Bach", "Hinton", "Yamamoto") 의 **언어/국가** 를 맞히는 문자 단위 분류기.

**데이터**: PyTorch 튜토리얼의 names dataset.
- 18 개 카테고리 (Arabic, Chinese, Czech, Dutch, English, French, German, Greek, Irish, Italian, Japanese, Korean, Polish, Portuguese, Russian, Scottish, Spanish, Vietnamese)
- 각 카테고리는 해당 언어권의 이름 리스트가 담긴 텍스트 파일

아래에서 자동으로 다운로드 + 압축 해제 + ASCII 정규화를 수행한 뒤, 카테고리별 80/20 train/test 로 분할합니다.


In [ ]:
# Download and preprocess names dataset (provided)
DATA_URL = 'https://download.pytorch.org/tutorial/data.zip'
DATA_DIR = './data'

if not os.path.isdir(os.path.join(DATA_DIR, 'names')):
    print(f'Downloading {DATA_URL} ...')
    os.makedirs(DATA_DIR, exist_ok=True)
    with urllib.request.urlopen(DATA_URL) as r:
        buf = io.BytesIO(r.read())
    with zipfile.ZipFile(buf) as z:
        z.extractall('.')
    print('Done.')
else:
    print('Dataset already present.')

ALL_LETTERS = string.ascii_letters + " .,;'"
N_LETTERS = len(ALL_LETTERS)


def unicode_to_ascii(s):
    return ''.join(
        c for c in unicodedata.normalize('NFD', s)
        if unicodedata.category(c) != 'Mn' and c in ALL_LETTERS
    )


def read_lines(filename):
    with open(filename, encoding='utf-8') as f:
        return [unicode_to_ascii(line.strip()) for line in f if line.strip()]


import glob
category_lines = {}
all_categories = []
for path in sorted(glob.glob(os.path.join(DATA_DIR, 'names', '*.txt'))):
    category = os.path.splitext(os.path.basename(path))[0]
    all_categories.append(category)
    category_lines[category] = read_lines(path)

N_CATEGORIES = len(all_categories)
print(f'{N_CATEGORIES} categories loaded')
print(f'  alphabet size: {N_LETTERS}')
print(f'  sample (Italian): {category_lines["Italian"][:5]}')
print(f'  sample (Korean):  {category_lines["Korean"][:5]}')


In [ ]:
# Encoding helpers (provided)
def letter_to_index(letter):
    return ALL_LETTERS.find(letter)


def line_to_tensor(line):
    """Returns a (T, 1, N_LETTERS) one-hot tensor for a name string."""
    tensor = torch.zeros(len(line), 1, N_LETTERS)
    for t, letter in enumerate(line):
        tensor[t, 0, letter_to_index(letter)] = 1
    return tensor


print('letter_to_index("a") =', letter_to_index('a'))
print('letter_to_index("Z") =', letter_to_index('Z'))
print('line_to_tensor("Bach") shape:', line_to_tensor('Bach').shape)


In [ ]:
# Quick dataset stats: samples per category + name length distribution
counts = [len(category_lines[c]) for c in all_categories]
all_lengths = [len(line) for c in all_categories for line in category_lines[c]]

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

ax = axes[0]
order = np.argsort(counts)[::-1]
ax.bar([all_categories[i] for i in order], [counts[i] for i in order], color='#1565C0')
ax.set_ylabel('Number of names')
ax.set_title('Samples per category')
plt.setp(ax.get_xticklabels(), rotation=45, ha='right', fontsize=9)
ax.grid(axis='y', alpha=0.3)

ax = axes[1]
ax.hist(all_lengths, bins=range(1, max(all_lengths) + 2), color='#388E3C', edgecolor='white')
ax.set_xlabel('Name length (chars)')
ax.set_ylabel('Count')
ax.set_title('Name length distribution')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout(); plt.show()

print(f'Total samples: {sum(counts):,}')
print(f'Mean name length: {np.mean(all_lengths):.1f}, max: {max(all_lengths)}')
print(f'Note: classes are imbalanced — Russian / English dominate.')


In [ ]:
# Train/test split (provided): per-category 80/20
random.seed(123)
train_examples = []   # list of (category, line)
test_examples = []
for cat in all_categories:
    names = list(category_lines[cat])
    random.shuffle(names)
    n_test = max(1, int(0.2 * len(names)))
    test_examples.extend([(cat, n) for n in names[:n_test]])
    train_examples.extend([(cat, n) for n in names[n_test:]])

# Build a per-category map of training examples for the random sampler
train_by_cat = defaultdict(list)
for cat, name in train_examples:
    train_by_cat[cat].append(name)

print(f'train: {len(train_examples):,}   test: {len(test_examples):,}')

cat_to_idx = {c: i for i, c in enumerate(all_categories)}


def random_train_example():
    """Sample one (cat, name) uniformly across categories from the train split."""
    cat = random.choice(all_categories)
    name = random.choice(train_by_cat[cat])
    target = torch.tensor([cat_to_idx[cat]], dtype=torch.long)
    return cat, name, target, line_to_tensor(name)


# quick check
random.seed(0)
for _ in range(3):
    cat, name, _, t = random_train_example()
    print(f'  category={cat:12s}  line={name:15s}  tensor shape={tuple(t.shape)}')


## 2. CharRNN 분류기 정의

`nn.RNN` 을 사용해 char-level 분류기를 만듭니다.
- 입력: `(T, 1, n_letters)` 모양의 one-hot 시퀀스 (한 이름 = 한 시퀀스)
- 모델: `nn.RNN` 한 번 통과 → 마지막 hidden state $h_T$ → `nn.Linear` 로 카테고리 logit
- 출력: shape `(1, n_categories)`

> batch size 1 로 한 이름씩 학습하는 단순한 루프를 사용합니다.


In [ ]:
class CharRNN(nn.Module):
    """
    nn.RNN 기반 char-level 분류기.
    구조: input (T, 1, n_letters) -> nn.RNN -> last hidden -> nn.Linear -> (1, n_categories)
    """
    def __init__(self, n_letters, hidden_size, n_categories):
        super().__init__()
        ############################################################################
        # TODO 1: __init__ 에서 RNN 모듈과 출력 Linear 를 정의하세요 (~2줄)              #
        # - self.rnn : nn.RNN(n_letters, hidden_size, nonlinearity='tanh')          #
        # - self.fc  : nn.Linear(hidden_size, n_categories)                         #
        ############################################################################
        self.rnn = nn.RNN(n_letters, hidden_size, nonlinearity='tanh')
        self.fc = nn.Linear(hidden_size, n_categories)
        ############################################################################
        #                             END OF YOUR CODE                             #
        ############################################################################

    def forward(self, x):
        ############################################################################
        # TODO 1 (cont.): forward 를 작성하세요 (~2줄)                                  #
        # - out, h_n = self.rnn(x)                                                   #
        # - return self.fc(h_n[-1])                                                  #
        # h_n 의 shape: (num_layers=1, batch=1, hidden_size) → 마지막을 잘라 fc에 통과     #
        ############################################################################
        out, h_n = self.rnn(x)
        return self.fc(h_n[-1])
        ############################################################################
        #                             END OF YOUR CODE                             #
        ############################################################################


In [ ]:
# Quick shape check + parameter listing
torch.manual_seed(0)
test_model = CharRNN(N_LETTERS, hidden_size=128, n_categories=N_CATEGORIES)
sample = line_to_tensor('Albert')
print(f'input shape:  {tuple(sample.shape)}')
out = test_model(sample)
print(f'output shape: {tuple(out.shape)}  (expect: (1, {N_CATEGORIES}))')

print('\nCharRNN parameters:')
total = 0
for name, p in test_model.named_parameters():
    print(f'  {name:25s}  shape={str(tuple(p.shape)):18s}  requires_grad={p.requires_grad}')
    total += p.numel()
print(f'  total trainable params: {total:,}')


## 3. 학습 루프

각 step:
1. train split 에서 무작위 (category, name) 한 쌍 추출
2. forward → cross-entropy loss → backward → Adam step
3. 일정 간격마다 running loss / running accuracy 기록

`train_classifier` 함수는 위 루프를 일반화해 RNN/LSTM 둘 다 동일하게 사용할 수 있게 만듭니다.
TODO 2 에서 한 step 의 forward/loss/backward/step 부분만 채웁니다.


In [ ]:
def train_classifier(model, n_iters=5000, log_every=250, lr=2e-3, seed=0):
    """
    Train a char classifier with batch_size=1 Adam.
    Returns: history dict with 'iter', 'loss', 'acc'.
    """
    torch.manual_seed(seed); random.seed(seed)
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = {'iter': [], 'loss': [], 'acc': []}

    running_loss = 0.0
    running_correct = 0
    for it in range(1, n_iters + 1):
        cat, line, target, x = random_train_example()
        x = x.to(device); target = target.to(device)

        ############################################################################
        # TODO 2: 한 step 의 forward / loss / backward / optimizer step (~5줄)            #
        # - logits = model(x)                                                         #
        # - loss = F.cross_entropy(logits, target)                                    #
        # - optimizer.zero_grad(); loss.backward(); optimizer.step()                  #
        ############################################################################
        logits = model(x)
        loss = F.cross_entropy(logits, target)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        ############################################################################
        #                             END OF YOUR CODE                             #
        ############################################################################

        running_loss += loss.item()
        pred = logits.argmax(dim=1)
        running_correct += int((pred == target).item())

        if it % log_every == 0:
            history['iter'].append(it)
            history['loss'].append(running_loss / log_every)
            history['acc'].append(running_correct / log_every)
            running_loss = 0.0
            running_correct = 0

    return history


In [ ]:
# Train CharRNN
torch.manual_seed(42)
rnn_model = CharRNN(N_LETTERS, hidden_size=128, n_categories=N_CATEGORIES)
print('Training CharRNN ...')
t0 = time.time()
rnn_history = train_classifier(rnn_model, n_iters=5000, log_every=250)
print(f'  done in {time.time() - t0:.1f}s   '
      f'final running loss={rnn_history["loss"][-1]:.3f}   '
      f'final running acc={rnn_history["acc"][-1]:.3f}')

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].plot(rnn_history['iter'], rnn_history['loss'], 'o-', color='#D32F2F')
axes[0].set_xlabel('iteration'); axes[0].set_ylabel('running loss')
axes[0].set_title('CharRNN training loss')
axes[0].grid(True, alpha=0.3)

axes[1].plot(rnn_history['iter'], rnn_history['acc'], 'o-', color='#D32F2F')
axes[1].set_xlabel('iteration'); axes[1].set_ylabel('running accuracy')
axes[1].set_title('CharRNN training (running) accuracy')
axes[1].grid(True, alpha=0.3)
plt.tight_layout(); plt.show()


## 4. Test 평가

지금까지의 running accuracy 는 학습 도중 본 sample 에서 측정한 값입니다.
실제 모델이 얼마나 잘 일반화되는지를 보려면 **학습에 쓰지 않은 test split** 으로 평가해야 합니다.

TODO 3 에서 test 평가 함수를 작성합니다.
이 함수는 정확도뿐 아니라 **모든 예측과 정답을 반환**해서, 그 결과로 confusion matrix / 카테고리별 정확도 / 길이별 정확도 등 다양한 분석을 할 수 있게 합니다.


In [ ]:
@torch.no_grad()
def evaluate(model, examples):
    """
    examples: list of (category, name)
    Returns:
        accuracy: float
        all_preds: list of int (predicted category index)
        all_targets: list of int (true category index)

    Hint:
        - model.eval() 호출
        - 각 (cat, name) 에 대해 line_to_tensor → model → argmax
        - 정답 / 예측 누적 후 accuracy 계산
    """
    ############################################################################
    # TODO 3: evaluate 를 구현하세요 (~8줄)                                          #
    ############################################################################
    model.eval()
    all_preds = []
    all_targets = []
    for cat, name in examples:
        x = line_to_tensor(name).to(device)
        logits = model(x)
        pred = int(logits.argmax(dim=1).item())
        all_preds.append(pred)
        all_targets.append(cat_to_idx[cat])
    correct = sum(p == t for p, t in zip(all_preds, all_targets))
    accuracy = correct / len(examples)
    ############################################################################
    #                             END OF YOUR CODE                             #
    ############################################################################
    return accuracy, all_preds, all_targets


rnn_acc, rnn_preds, rnn_targets = evaluate(rnn_model, test_examples)
print(f'CharRNN test accuracy: {rnn_acc:.4f}  ({len(test_examples)} examples)')


In [ ]:
# Confusion matrix + per-category accuracy for CharRNN (provided)
def confusion_matrix(preds, targets, n_classes):
    cm = np.zeros((n_classes, n_classes), dtype=int)
    for p, t in zip(preds, targets):
        cm[t, p] += 1
    return cm


def per_category_accuracy(cm):
    row_sums = cm.sum(axis=1)
    diag = np.diag(cm)
    return np.where(row_sums > 0, diag / np.maximum(row_sums, 1), 0.0)


cm_rnn = confusion_matrix(rnn_preds, rnn_targets, N_CATEGORIES)
per_acc_rnn = per_category_accuracy(cm_rnn)

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))

# Confusion matrix heatmap (row-normalized)
ax = axes[0]
cm_norm = cm_rnn / np.maximum(cm_rnn.sum(axis=1, keepdims=True), 1)
im = ax.imshow(cm_norm, cmap='Blues', vmin=0, vmax=1)
ax.set_xticks(range(N_CATEGORIES)); ax.set_yticks(range(N_CATEGORIES))
ax.set_xticklabels(all_categories, rotation=60, ha='right', fontsize=8)
ax.set_yticklabels(all_categories, fontsize=8)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title('CharRNN — confusion matrix (row-normalized)')
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

# Per-category accuracy
ax = axes[1]
order = np.argsort(per_acc_rnn)[::-1]
ax.bar(range(N_CATEGORIES),
       [per_acc_rnn[i] for i in order],
       color='#1565C0')
ax.set_xticks(range(N_CATEGORIES))
ax.set_xticklabels([all_categories[i] for i in order], rotation=60, ha='right', fontsize=8)
ax.set_ylabel('Test accuracy')
ax.set_title('CharRNN — per-category test accuracy')
ax.axhline(rnn_acc, color='black', linestyle='--', lw=1, label=f'overall = {rnn_acc:.3f}')
ax.set_ylim(0, 1.0)
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout(); plt.show()

# Look at the strongest off-diagonal confusions
off = cm_norm.copy()
np.fill_diagonal(off, 0)
top_idx = np.dstack(np.unravel_index(np.argsort(off.ravel())[::-1], off.shape))[0][:5]
print('\nTop off-diagonal confusions (true -> predicted):')
for (i, j) in top_idx:
    print(f'  {all_categories[i]:12s} -> {all_categories[j]:12s}   row-frac={off[i,j]:.2f}')


## 5. CharLSTM 과 비교

`nn.RNN` 을 `nn.LSTM` 으로 한 줄만 바꾸면 됩니다.
주의할 점은 `nn.LSTM` 의 forward 가 `(output, (h_n, c_n))` 처럼 hidden state 두 개를 묶어서 돌려준다는 것입니다.

같은 hidden size, 같은 학습 설정, 같은 시드로 비교 학습한 뒤 두 모델을 나란히 비교합니다.


In [ ]:
class CharLSTM(nn.Module):
    """
    nn.LSTM 기반 char-level 분류기.
    CharRNN 과 동일 구조이지만 nn.RNN 대신 nn.LSTM 사용.
    forward 에서는 (h_n, c_n) tuple 중 h_n 만 분류에 사용합니다.
    """
    def __init__(self, n_letters, hidden_size, n_categories):
        super().__init__()
        ############################################################################
        # TODO 4: __init__ 에서 LSTM 모듈과 출력 Linear 를 정의하세요 (~2줄)              #
        # - self.lstm : nn.LSTM(n_letters, hidden_size)                              #
        # - self.fc   : nn.Linear(hidden_size, n_categories)                         #
        ############################################################################
        self.lstm = nn.LSTM(n_letters, hidden_size)
        self.fc = nn.Linear(hidden_size, n_categories)
        ############################################################################
        #                             END OF YOUR CODE                             #
        ############################################################################

    def forward(self, x):
        ############################################################################
        # TODO 4 (cont.): forward 를 작성하세요 (~2줄)                                  #
        # - out, (h_n, c_n) = self.lstm(x)                                            #
        # - return self.fc(h_n[-1])                                                   #
        ############################################################################
        out, (h_n, c_n) = self.lstm(x)
        return self.fc(h_n[-1])
        ############################################################################
        #                             END OF YOUR CODE                             #
        ############################################################################


# Train CharLSTM with the same settings
torch.manual_seed(42)
lstm_model = CharLSTM(N_LETTERS, hidden_size=128, n_categories=N_CATEGORIES)
print('Training CharLSTM ...')
t0 = time.time()
lstm_history = train_classifier(lstm_model, n_iters=5000, log_every=250)
print(f'  done in {time.time() - t0:.1f}s   '
      f'final running loss={lstm_history["loss"][-1]:.3f}   '
      f'final running acc={lstm_history["acc"][-1]:.3f}')

# Evaluate on the same test split using TODO 3
lstm_acc, lstm_preds, lstm_targets = evaluate(lstm_model, test_examples)
print(f'\nCharRNN  test accuracy: {rnn_acc:.4f}')
print(f'CharLSTM test accuracy: {lstm_acc:.4f}')


In [ ]:
# Side-by-side: training curves AND confusion matrices for the two models
fig = plt.figure(figsize=(15, 9))

# Top row: loss + accuracy curves
ax = fig.add_subplot(2, 2, 1)
ax.plot(rnn_history['iter'], rnn_history['loss'], 'o-', color='#D32F2F', label='CharRNN')
ax.plot(lstm_history['iter'], lstm_history['loss'], 's-', color='#1565C0', label='CharLSTM')
ax.set_xlabel('iteration'); ax.set_ylabel('running loss')
ax.set_title('Training loss')
ax.legend(); ax.grid(True, alpha=0.3)

ax = fig.add_subplot(2, 2, 2)
ax.plot(rnn_history['iter'], rnn_history['acc'], 'o-', color='#D32F2F', label='CharRNN (running)')
ax.plot(lstm_history['iter'], lstm_history['acc'], 's-', color='#1565C0', label='CharLSTM (running)')
ax.axhline(rnn_acc, color='#D32F2F', linestyle=':', lw=1.2, label=f'CharRNN test = {rnn_acc:.3f}')
ax.axhline(lstm_acc, color='#1565C0', linestyle=':', lw=1.2, label=f'CharLSTM test = {lstm_acc:.3f}')
ax.set_xlabel('iteration'); ax.set_ylabel('accuracy')
ax.set_title('Training (running) accuracy + final test accuracy')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

# Bottom row: confusion matrices
cm_lstm = confusion_matrix(lstm_preds, lstm_targets, N_CATEGORIES)
for ax_idx, (cm, title) in enumerate([(cm_rnn, 'CharRNN'), (cm_lstm, 'CharLSTM')]):
    ax = fig.add_subplot(2, 2, 3 + ax_idx)
    cm_norm = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)
    im = ax.imshow(cm_norm, cmap='Blues', vmin=0, vmax=1)
    ax.set_xticks(range(N_CATEGORIES)); ax.set_yticks(range(N_CATEGORIES))
    ax.set_xticklabels(all_categories, rotation=60, ha='right', fontsize=7)
    ax.set_yticklabels(all_categories, fontsize=7)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    ax.set_title(f'{title} — confusion matrix')
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.tight_layout(); plt.show()

print(f'\nFinal:  CharRNN test acc = {rnn_acc:.4f}    CharLSTM test acc = {lstm_acc:.4f}')


## 6. 더 들여다보기

여기서는 두 가지 추가 분석을 합니다.
1. **이름 길이별 정확도** — RNN 은 시퀀스 길이가 길어질수록 gradient 가 약해지는 경향이 있었으니, 긴 이름에서 LSTM 이 더 잘 할 가능성이 있습니다. test set 을 길이 bucket 으로 나눠 두 모델의 정확도를 비교합니다.
2. **두 모델의 의견 불일치** — RNN 과 LSTM 이 서로 다르게 예측한 test 예제를 골라 일부 출력합니다. 어느 쪽이 맞고 어느 쪽이 틀렸는지를 보면 두 모델의 강점/약점을 직관적으로 알 수 있습니다.


In [ ]:
# Length-binned test accuracy: bucket test names by length and compute per-bucket accuracy
length_bins = [(1, 4), (5, 6), (7, 8), (9, 10), (11, 20)]


def acc_by_length(preds, targets):
    """Returns dict {bin_label: (count, accuracy)}."""
    out = {}
    for lo, hi in length_bins:
        idxs = [k for k, (cat, name) in enumerate(test_examples) if lo <= len(name) <= hi]
        if not idxs:
            out[f'{lo}-{hi}'] = (0, 0.0)
            continue
        right = sum(preds[k] == targets[k] for k in idxs)
        out[f'{lo}-{hi}'] = (len(idxs), right / len(idxs))
    return out


rnn_by_len = acc_by_length(rnn_preds, rnn_targets)
lstm_by_len = acc_by_length(lstm_preds, lstm_targets)

labels = list(rnn_by_len.keys())
rnn_vals = [rnn_by_len[k][1] for k in labels]
lstm_vals = [lstm_by_len[k][1] for k in labels]
counts = [rnn_by_len[k][0] for k in labels]

x = np.arange(len(labels))
w = 0.38
fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(x - w / 2, rnn_vals, w, color='#D32F2F', label='CharRNN')
ax.bar(x + w / 2, lstm_vals, w, color='#1565C0', label='CharLSTM')
for i, c in enumerate(counts):
    ax.text(i, max(rnn_vals[i], lstm_vals[i]) + 0.02, f'n={c}',
            ha='center', fontsize=9, color='#555')
ax.set_xticks(x); ax.set_xticklabels(labels)
ax.set_xlabel('Name length (chars)')
ax.set_ylabel('Test accuracy')
ax.set_title('Test accuracy by name length')
ax.set_ylim(0, 1.05)
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.show()

print('\n[ Test accuracy by length ]')
print(f'  {"length":8s}  {"n":>5s}   {"RNN":>7s}   {"LSTM":>7s}')
for k in labels:
    n, _ = rnn_by_len[k]
    print(f'  {k:8s}  {n:5d}   {rnn_by_len[k][1]:7.3f}   {lstm_by_len[k][1]:7.3f}')


In [ ]:
# Find names where the two models disagree on the test set
disagreements = []
for k, (cat, name) in enumerate(test_examples):
    rp = rnn_preds[k]
    lp = lstm_preds[k]
    if rp != lp:
        disagreements.append((cat, name, rp, lp))

n_show = 12
print(f'Total disagreements: {len(disagreements)}  /  test = {len(test_examples)}')
print(f'\nShowing {min(n_show, len(disagreements))} examples:')
print(f'  {"true":12s}  {"name":15s}  {"RNN pred":15s}  {"LSTM pred":15s}  who_correct')
print('  ' + '-' * 75)
random.seed(7)
sample = random.sample(disagreements, min(n_show, len(disagreements)))
rnn_wins = lstm_wins = 0
for cat, name, rp, lp in sample:
    rnn_correct = (all_categories[rp] == cat)
    lstm_correct = (all_categories[lp] == cat)
    if rnn_correct: rnn_wins += 1
    if lstm_correct: lstm_wins += 1
    who = 'RNN' if rnn_correct else ('LSTM' if lstm_correct else 'neither')
    print(f'  {cat:12s}  {name:15s}  {all_categories[rp]:15s}  {all_categories[lp]:15s}  {who}')

# Tally over ALL disagreements
total_rnn = sum(all_categories[rp] == cat for cat, _, rp, _ in disagreements)
total_lstm = sum(all_categories[lp] == cat for cat, _, _, lp in disagreements)
print(f'\nAcross all disagreements:  RNN correct {total_rnn}  /  LSTM correct {total_lstm}')


# Top-k predictions for a few hand-picked names
@torch.no_grad()
def predict_topk(model, line, k=3):
    model.eval()
    x = line_to_tensor(line).to(device)
    logits = model(x)
    probs = F.softmax(logits, dim=1)[0]
    top_p, top_i = probs.topk(k)
    return [(all_categories[i], float(p)) for p, i in zip(top_p.cpu(), top_i.cpu())]


test_names = ['Bach', 'Hinton', 'Yamamoto', 'Kim', 'Garcia', 'Schmidt', 'Nguyen', 'Romanov']
print(f'\n{"name":12s}  {"RNN top-1":25s}  {"LSTM top-1":25s}')
print('-' * 70)
for name in test_names:
    r = predict_topk(rnn_model, name, k=1)[0]
    l = predict_topk(lstm_model, name, k=1)[0]
    print(f'{name:12s}  {r[0]:15s} ({r[1]:.2f})  {l[0]:15s} ({l[1]:.2f})')


## 정리

### 이번 실습에서 한 일

| 단계 | 내용 |
|------|------|
| 데이터 | PyTorch tutorial dataset 자동 다운로드 → unicode→ASCII 정규화 → 카테고리당 80/20 train/test 분할 |
| 모델 | `CharRNN` (`nn.RNN`) / `CharLSTM` (`nn.LSTM`) — 마지막 hidden state → linear 분류 헤드 |
| 학습 | Adam (lr=2e-3), batch_size=1, 5000 iter 으로 두 모델을 같은 시드에서 비교 |
| 평가 | `evaluate(model, test_examples)` 로 test accuracy + 모든 prediction 수집 |
| 분석 | (1) confusion matrix (row-normalized) (2) per-category accuracy (3) 길이별 accuracy (4) RNN vs LSTM disagreement |

### Day4 전체 요약

| 노트북 | 핵심 내용 |
|--------|----------|
| **260415-1** | numpy 로 RNN / LSTM / GRU forward 직접 구현 + PyTorch 와 수치 비교 + 세 cell 의 gradient 흐름 비교 |
| **260415-2** | PyTorch `nn.RNN` / `nn.LSTM` 으로 char-level 이름 분류 + train/test 평가 + confusion matrix / 길이별 / disagreement 분석 |
